In [1]:
import numpy as np
from pathlib import Path
import sys
import os
import xarray as xr

In [10]:
root_dir = Path(os.getcwd()).parent.parent.parent
data_dir = root_dir / 'data' / 'ca_imaging'

# Add the model directory to Python path
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.data_pipeline.preprocessing.preprocess import normalize99

Load original data

In [3]:
filenames = [data_dir / "cells_train.npz",
              data_dir / "cells_test.npz"]

cells_train = np.load(data_dir / 'cells_train.npz', allow_pickle=True)['arr_0'].item()
cells_test = np.load(data_dir / 'cells_test.npz', allow_pickle=True)['arr_0'].item()

images_train = np.array(cells_train['imgs']).transpose(0, 3, 1, 2)
masks_train = np.array(cells_train['masks'])
images_test = np.array(cells_test['imgs']).transpose(0, 3, 1, 2)
masks_test = np.array(cells_test['masks'])

Concatenate training and test data

In [4]:
images = np.concatenate([images_train, images_test], axis = 0)
masks = np.concatenate([masks_train, masks_test], axis = 0)
# make cell present/no cell present labels for each pixel
labels = (masks > 0).astype(np.longlong)

Pad images and labels so that their dimensions are divisible by two for the Unet network.

In [27]:
# @markdown Padding code for test images

def pad_image_ND(img0, div=16, extra=1):
  """ pad image for test-time so that its dimensions are a multiple of 16 (2D or 3D)

  Parameters
  -------------
  img0: ND-array
      image of size [nchan (x Lz) x Ly x Lx]
  div: int (optional, default 16)

  Returns
  --------------
  I: ND-array
      padded image
  slices: tuple, int
      range of pixels in I corresponding to img0
  """
  Lpad = int(div * np.ceil(img0.shape[-2] / div) - img0.shape[-2])
  xpad1 = extra * div//2 + Lpad//2
  xpad2 = extra * div//2 + Lpad - Lpad//2
  Lpad = int(div * np.ceil(img0.shape[-1] / div) - img0.shape[-1])
  ypad1 = extra * div//2 + Lpad//2
  ypad2 = extra * div//2 + Lpad - Lpad//2

  if img0.ndim > 2:
    pads = np.array([[0, 0], [xpad1, xpad2], [ypad1, ypad2]])
  else:
    pads = np.array([[xpad1, xpad2], [ypad1, ypad2]])

  I = np.pad(img0, pads, mode='constant')

  Ly, Lx = img0.shape[-2:]
  ysub = np.arange(xpad1, xpad1 + Ly)
  xsub = np.arange(ypad1, ypad1 + Lx)
  slc = [slice(0, img0.shape[n] + 1) for n in range(img0.ndim)]
  #slc[-3] = slice(0, img0.shape[-3] + 1)
  slc[-2] = slice(ysub[0], ysub[-1] + 1)
  slc[-1] = slice(xsub[0], xsub[-1] + 1)
  slc = tuple(slc)

  return I#, slc

In [28]:
images_padded, labels_padded = [], []
for image, label in zip(images, labels):
    images_padded.append(pad_image_ND(image, 8))
    labels_padded.append(pad_image_ND(label, 8))

images_padded = np.array(images_padded)
labels_padded = np.array(labels_padded)

Normalize image data

In [31]:
images_padded = np.array([normalize99(img) for img in images_padded])

Create xarray DataArrays for images and labels and put them together in xarray Dataset.

In [32]:
images_xarr = xr.DataArray(
        data=images_padded,
        dims=['image_nr', 'staining', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(images_padded.shape[0])),
                'staining' : ('staining', ['cytoplasm', 'nuclear'])},
        name='ca_images',
        attrs={
            'description': 'Calcium imaging data with two kinds of staining',
            'notes': 'Cytoplasm means whole cell stained, nuclear means only nucleus of cells stained.'
        }
    )

labels_xarr = xr.DataArray(
        data=labels_padded,
        dims=['image_nr', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(images_padded.shape[0]))},
        name='cell_labels',
        attrs={
            'description': 'Cell labels',
            'notes': '1 means there is a cell at pixel, 0 means there is no cell at pixel.'
        }
    )

ds = xr.Dataset({
    'ca_image_data': images_xarr,
    'cell_labels': labels_xarr
})
ds

<xarray.Dataset> Size: 297MB
Dimensions:        (image_nr: 91, staining: 2, height: 392, width: 520)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) float32 148MB 0.0 ... 0.0
    cell_labels    (image_nr, height, width) int64 148MB 0 0 0 0 0 ... 0 0 0 0 0

Write dataset to file

In [33]:
ds.to_netcdf(data_dir / 'cells_data_all.nc')

Check saved file

In [34]:
dataset_check = xr.load_dataset(data_dir / 'cells_data_all.nc')

dataset_check

<xarray.Dataset> Size: 297MB
Dimensions:        (image_nr: 91, staining: 2, height: 392, width: 520)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) float32 148MB 0.0 ... 0.0
    cell_labels    (image_nr, height, width) int64 148MB 0 0 0 0 0 ... 0 0 0 0 0